In [ ]:
from ntscraper import Nitter
import pandas as pd
import tqdm
import json
import numpy as np
import time
import json
import requests
from bs4 import BeautifulSoup


#get bad tweets

In [ ]:
def sc(term, mode, number):
    scraper = Nitter()
    
    tweets = scraper.get_tweets(term, mode=mode, number=number, exclude=['videos', 'native_video', 'links', 'images', 'pro_video'], language='fa')
    
    tweet_list = [tw['text'] for tw in tweets['tweets']]
    
    return tweet_list

In [ ]:
existing_data = {}
try:
    with open('Only_Persian_Offensive_lexicon.json', 'r', encoding='utf-8') as existing_file:
        existing_data = json.load(existing_file)
except FileNotFoundError:
    pass

swearwords = existing_data.get('word', [])
len(swearwords)  

In [ ]:
all_bad_tweets = []
i=0
for bad_word in swearwords:
    print(i)
    print('--------------- getting tweets containing "' + bad_word + '"------------------')
    try:
        bad_tweet_list = sc(bad_word, 'term', 10)
        for tweet in bad_tweet_list:
            all_bad_tweets.append([bad_word,tweet])   
        time.sleep(30) # avoids Nitter's 'Max retries exceeded' error
    except Exception as err:  
        print (err)        
        time.sleep(60) # back off after an error, then continue with the next word
        continue
    i += 1

In [ ]:
import re
# remove #, commas, new lines, retweet markers and @mentions
all_bad_tweets_clean= [[tweet[0],tweet[1].replace("#" , "").replace("," , " ").replace("\n" , '')] for tweet in all_bad_tweets ]
all_bad_tweets_clean= [[tweet[0],re.sub('RT @[^\s]+','',tweet[1])] for tweet in all_bad_tweets_clean ]
all_bad_tweets_clean= [[tweet[0],re.sub('@[^\s]+','',tweet[1])] for tweet in all_bad_tweets_clean ]

In [ ]:
bad_tweets_df = pd.DataFrame(all_bad_tweets_clean, columns=['badword', 'text'])
bad_tweets_df['label'] = bad_tweets_df.apply(lambda x: 1, axis=1)
bad_tweets_df.columns= ['badword','text' , 'label']

bad_tweets_df.to_csv('AllBad2sidewordTweetsthird.csv', mode='w', header=True, index=True,  encoding='utf-8')

bad_tweets_df.head(900)